In [ ]:
%reload_ext autoreload
%autoreload 2
import os, sys
import torch
import numpy as np
import pandas as pd

import torch.utils.data as data_utils
from functools import partial
from torch.utils.data import DataLoader

from searchspace import NDS
import pycls.datasets.loader as loader

# search space available on https://dl.fbaipublicfiles.com/nds/data.zip
# https://github.com/facebookresearch/nds?tab=readme-ov-file


In [ ]:
# setup & hyperparameters
_selected_ss = 3
RANDOM_SEED = 4 
# export
export_into_csv = True
reeval_csv = False

# search space sepectific
search_spaces = ['DARTS', 'ENAS', 'PNAS', 'NASNet', 'Amoeba']
search_space_str = search_spaces[_selected_ss]
# batch size / dataset shape
BATCH_SIZE = 64
_input_shape = (BATCH_SIZE, 3, 16, 16)
coeff = ["Spearman's", "Kendall's"]

# seed + generator
torch.manual_seed(RANDOM_SEED)
torch.cuda.manual_seed(RANDOM_SEED)
rng_generator = np.random.default_rng(seed=RANDOM_SEED)

# set up device
device = (
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

# TODO change the path where you unpacked your zip file (Download at https://dl.fbaipublicfiles.com/nds/data.zip)
path = "/mnt/localdata/global/common/acd_datasets/nas/NDS"
# path = "./data"
nds_searchpace_c = NDS(search_spaces[_selected_ss], path=path)
size_nds_ss = nds_searchpace_c.__len__()
_n_models = 500

# csv export
FOLDER = "../experiments/NDS_{}_cifar10_ch/".format(search_space_str)
os.makedirs(f"{FOLDER}",exist_ok=True)

In [ ]:
# load models and acc
_models, accuracy = [], []
instances = rng_generator.choice(size_nds_ss, size=(_n_models), replace=False)

for i in range(0, _n_models):
    model = nds_searchpace_c.get_network(i).to(device)
    acc = nds_searchpace_c.get_final_accuracy(i)
    _models.append(model)
    accuracy.append(acc)

# correlation
from scipy.stats import kendalltau, spearmanr
corr_coeffs = [
    partial(spearmanr, b=accuracy, nan_policy='omit'),
    partial(kendalltau, y=accuracy, nan_policy='omit'),
]

In [ ]:
# load dataset -> cifar10
data_path = os.getcwd()+"/../datasets/cifar10/cifar-10-batches-py/"
train_data, train_loader = loader._construct_loader('cifar10', 'train', BATCH_SIZE, True, True, data_path)

# get shortend trainloader for Zico
dataset = data_utils.Subset(train_data, torch.arange(4 * BATCH_SIZE))
train_loader_shorted = DataLoader(dataset=dataset, batch_size=BATCH_SIZE, shuffle=True)

In [ ]:
# declarations and definitions for the DAEP
bez = ['a', 'b', 'c', 'd', 'e', 'f', 'g', 'ECP','all']    # used for naming of export files, titles (x/y-axis) of plots, ...
EP_titles = ["EP_a", "EP_b", "EP_c", "EP_d", "EP_e", "EP_f", "EP_g", "A_ECP", "ALL", "test"] #,"SNIP"
gr_truth = accuracy

# config = [[None, 0], [None, 1], [None, 2], [None, 3], [None, 4], [None, 5], [None, 6], [None, 7]]
config = [[None, 8]]
EP_titles_upd = [EP_titles[conf[1]] for conf in config] 

n_eval = 150 #150 
cand_rank = 7
n_min = 10 # window size?
eval_mode = 8  
threshold = -1

FILENAME_EXPERIMENT_EP = "benchmark_DAEP_class_acc_cr{}_m{}_tr{}_numMod{}_seed{}".format(cand_rank, eval_mode, n_eval, _n_models, RANDOM_SEED)
os.makedirs(f"{FOLDER}",exist_ok=True)
path = os.getcwd() + "/" + FOLDER + FILENAME_EXPERIMENT_EP
file_there = os.path.isfile(path + "_dict.csv")

In [ ]:
import ast
sys.path.append("..")
from ensemble_proxy_ch import Ensemble_Proxy

proxy_dicts, EP_scores_all, EP_timing_all, break_points_all, EP_hyperpara_all, EP_weights_all, EP_full_history_all = [], [], [], [], [], [], []
EP_class = Ensemble_Proxy(_models, None, _input_shape, verbose=False)

if os.path.isfile(path + "_dict.csv") and not reeval_csv:
    df_1 = pd.read_csv(path + "_dict.csv")
    df_2 = pd.read_csv(path + "_scores.csv")
    df_3 = pd.read_csv(path + "_bp.csv")
    df_4 = pd.read_csv(path + "_weights.csv")
    df_5 = pd.read_csv(path + "_hyperpara.csv")
    df_6 = pd.read_csv(path + "_full_Whistory.csv")
    for key in EP_titles_upd:
        proxy_dicts.append(ast.literal_eval(df_1[key+"_dict"].values[0]))
        EP_scores_all.append(df_2[key+"_scores"].values.tolist())
        EP_timing_all.append(df_2[key+"_time"].values.tolist())
        break_points_all.append(df_3[key+"_bp"].values[0])
        EP_weights_all.append(ast.literal_eval(df_4[key+"_weight"].values[0]))
        EP_hyperpara_all.append(ast.literal_eval(df_5[key+"_hyper"].values[0]))
        EP_full_history_all.append(ast.literal_eval(df_6[key+"_full_Whis"].values[0]))
else:
    for x in config:
        EP_class.set_config(x)
        EP_class.reset_weights()
        proxy_dict, EP_scores, EP_timing, break_points, weight_history_all = EP_class.adapt_ensemble_proxy(model_acc=gr_truth, n_min=n_min, weight_config=None, train_loader=train_loader_shorted, n_eval=n_eval, nats=False)
        proxy_dicts.append(proxy_dict)
        EP_scores_all.append(EP_scores)
        EP_timing_all.append(EP_timing)
        break_points_all.append(break_points)
        EP_hyperpara_all.append(EP_class.get_hyperpara())
        EP_weights_all.append(EP_class._sum_weights.tolist())
        EP_full_history_all.append(weight_history_all)

In [ ]:
sys.path.append("..")
from ensemble_proxy_ch import Ensemble_Proxy
# stopping criteria evaluation
EP_class = Ensemble_Proxy(_models, None, _input_shape, verbose=False)
select_config = 0
eval_scores=True
EP_class.set_config(config[select_config])
proxy_dict, EP_scores, break_points, weight_history_all = EP_class.eval_stopping_criteria(model_acc=gr_truth, n_min=n_min, weight_config=None, train_loader=train_loader_shorted, n_eval=n_eval, nats=False)
print(f"Breakpoint at {break_points}")

In [ ]:
from plotting.plotter import plot_weights
_dataset_string="NDS"
# print(weight_history_all)
# print(len(transf_weights))
# print(len(transf_weights[0]))
plot_weights([weight_history_all], config, bez, save_result=False, n_min=n_min, sss=True, dataset_str=_dataset_string, seed=RANDOM_SEED)

In [ ]:
if eval_scores:
    _, EP_scores_stop_crit= EP_class.eval_scores(train_loader_shorted, nats=False)
    print("\nconfig_pos: ", config[select_config][1])

In [ ]:
print(EP_scores_stop_crit)
print(f"Breakpoint: {break_points}")
tmp_str=""
for i in range(len(corr_coeffs)):
    print("{}: \t".format(coeff[i]), corr_coeffs[i](EP_scores_stop_crit).statistic if i < (len(corr_coeffs)-1) else corr_coeffs[i](EP_scores_stop_crit))

    tmp_str += "{}: \t {} \n".format(coeff[i], corr_coeffs[i](EP_scores_stop_crit).statistic if i < (len(corr_coeffs)-1) else corr_coeffs[i](EP_scores_stop_crit))



# Update filename for breakpoints after stopping criteria        
FILENAME_EXPERIMENT_EP = "benchmark_EP_class_{}_cr{}_m{}_tr{}_numMod{}_seed{}".format("accuracy", cand_rank, eval_mode, break_points, _n_models, RANDOM_SEED)
path = os.getcwd() + "/" + FOLDER + FILENAME_EXPERIMENT_EP
print(f"PATH: {path}")
with open (path+"_statistics.txt", "w+") as f:
    f.write(tmp_str)  

In [ ]:
if (export_into_csv and not os.path.isfile(path + "_dict.csv")) or reeval_csv:
    df_1 = pd.DataFrame(columns=[t + "_dict" for t in EP_titles_upd])
    df_1.loc[0] = proxy_dicts
    df_1.to_csv(path + "_dict.csv", index=False)

    EP_export_titles = np.concatenate(([t + "_scores" for t in EP_titles_upd], [t + "_time" for t in EP_titles_upd]))
    EP_export_data = np.concatenate((EP_scores_all, EP_timing_all)).T
    df_2 = pd.DataFrame(EP_export_data, columns=EP_export_titles)
    df_2.to_csv(path + "_scores.csv", index=False)

    df_3 = pd.DataFrame(columns=[t + "_bp" for t in EP_titles_upd])
    df_3.loc[0] = break_points_all
    df_3.to_csv(path + "_bp.csv")

    df_4 = pd.DataFrame(columns=[t + "_weight" for t in EP_titles_upd])
    df_4.loc[0] = EP_weights_all
    df_4.to_csv(path + "_weights.csv")

    df_5 = pd.DataFrame(columns=[t + "_hyper" for t in EP_titles_upd])
    df_5.loc[0] = EP_hyperpara_all
    df_5.to_csv(path + "_hyperpara.csv")

    df_6 = pd.DataFrame(columns=[t + "_full_Whis" for t in EP_titles_upd])
    df_6.loc[0] = EP_full_history_all
    df_6.to_csv(path + "_full_Whistory.csv")

In [ ]:
# eval correlation
select_config = 0
EP_class.reset_weights()
EP_class.set_weights(EP_weights_all[select_config], EP_hyperpara_all[select_config])
scores_reeval = [EP_class.reeval_sums_w_sparsification_explo(scores=proxy_dicts[select_config])]

for j in range(len(config)):
    print("\nconfig_pos: ", config[j][1])
    for i in range(len(corr_coeffs)):
        print("{}: \t".format(coeff[i]), corr_coeffs[i](scores_reeval[j]).statistic)